# Guía: preparar los datos de farmacias para InfluxDB (Jupyter)

**Objetivo (Tarea 1):** cargar el Excel, transformarlo y dejarlo listo para guardarlo en InfluxDB.

Recuerda cómo guarda los datos InfluxDB: cada fila necesita una **fecha** (`time`), unas **etiquetas** (tags, texto para filtrar) y unos **valores** (fields, números).

| Columna | Tipo en InfluxDB |
|---|---|
| Fecha | time |
| Country, Category, PromoFlag | tags |
| ventas, unidades, ingresos, costes, margen | fields |

**Pasos:** 1) cargar y unir, 2) crear la fecha, 3) sumar por día, 4) sumar por mes, 5) comprobar, 6) guardar CSV, 7) subir a InfluxDB (opcional).

## Paso 1. Cargar las hojas del Excel y unirlas


In [3]:
import pandas as pd
import numpy as np

RUTA = "/mnt/c/clase/Reto 0/pharmacy-sales-bigdata/Data/Pharmacy_data.xlsx" 

Datos = pd.read_excel(RUTA, sheet_name=None)
for nombre, df in Datos.items():
    print(nombre, df.shape)

FactSales (62139, 9)
DimDate (731, 7)
DimPharmacy (120, 10)
DimProduct (220, 11)


In [4]:
fact = Datos["FactSales"]
farmacia = Datos["DimPharmacy"]
producto = Datos["DimProduct"]

data = (
    fact
    .merge(farmacia[["PharmacyID", "Country"]], on="PharmacyID", how="left", validate="many_to_one")
    .merge(producto[["ProductID", "Category"]], on="ProductID", how="left", validate="many_to_one")
)
data.head()

,SalesID,DateKey,PharmacyID,ProductID,UnitsSold,RevenueEUR,CostEUR,MarginEUR,PromoFlag,Country,Category
0,S0000001,20240101,PH0002,PR0099,5,128.08,87.55,40.53,No,Austria,Personal Care
1,S0000002,20240101,PH0004,PR0156,4,51.89,34.32,17.57,No,Poland,Wellness
2,S0000003,20240101,PH0007,PR0004,20,317.73,199.53,118.20,No,Germany,Personal Care
3,S0000004,20240101,PH0009,PR0075,6,90.34,67.49,22.85,No,Germany,OTC
4,S0000005,20240101,PH0010,PR0154,2,160.21,124.99,35.22,Yes,Belgium,Medical Devices


## Paso 2. Crear la columna de fecha

`DateKey` viene como número (por ejemplo `20240315`). InfluxDB necesita una fecha de verdad, y mejor con zona horaria (usamos UTC).

In [5]:
data["Fecha"] = pd.to_datetime(data["DateKey"].astype(str), format="%Y%m%d").dt.tz_localize("UTC")
data[["DateKey", "Fecha"]].head()

,DateKey,Fecha
0,20240101,2024-01-01 00:00:00+00:00
1,20240101,2024-01-01 00:00:00+00:00
2,20240101,2024-01-01 00:00:00+00:00
3,20240101,2024-01-01 00:00:00+00:00
4,20240101,2024-01-01 00:00:00+00:00


## Paso 3. Sumar las ventas por día

**Esto es lo más importante.** En InfluxDB, si dos filas tienen el mismo measurement, las mismas tags y la misma fecha, la segunda **sobrescribe** a la primera. Tus ventas se repiten mucho en un mismo día, país, categoría y promoción, así que las sumamos para que cada combinación sea única.

In [6]:
columnas_tag = ["Country", "Category", "PromoFlag"]

diario = (
    data.groupby(["Fecha"] + columnas_tag, as_index=False)
    .agg(ventas=("SalesID", "count"),
         unidades=("UnitsSold", "sum"),
         ingresos_eur=("RevenueEUR", "sum"),
         costes_eur=("CostEUR", "sum"),
         margen_eur=("MarginEUR", "sum"))
)
print(diario.shape)
diario.head()

(29435, 9)


,Fecha,Country,Category,PromoFlag,ventas,unidades,ingresos_eur,costes_eur,margen_eur
0,2024-01-01 00:00:00+00:00,Austria,Medical Devices,No,1,1,24.78,17.38,7.40
1,2024-01-01 00:00:00+00:00,Austria,OTC,No,5,58,522.57,378.83,143.74
2,2024-01-01 00:00:00+00:00,Austria,Personal Care,No,1,5,128.08,87.55,40.53
3,2024-01-01 00:00:00+00:00,Austria,Prescription,Yes,1,3,36.09,30.82,5.27
4,2024-01-01 00:00:00+00:00,Austria,Wellness,No,4,37,560.83,352.36,208.47


## Paso 4. Sumar las ventas por mes

Es el mismo cálculo, pero la fecha es el primer día de cada mes. Irá al segundo bucket (`ventas_mensuales`).

In [7]:
data["Mes"] = data["Fecha"].dt.tz_localize(None).dt.to_period("M").dt.to_timestamp().dt.tz_localize("UTC")

mensual = (
    data.groupby(["Mes"] + columnas_tag, as_index=False)
    .agg(ventas=("SalesID", "count"),
         unidades=("UnitsSold", "sum"),
         ingresos_eur=("RevenueEUR", "sum"),
         costes_eur=("CostEUR", "sum"),
         margen_eur=("MarginEUR", "sum"))
    .rename(columns={"Mes": "Fecha"})
)
print(mensual.shape)
mensual.head()

(1908, 9)


,Fecha,Country,Category,PromoFlag,ventas,unidades,ingresos_eur,costes_eur,margen_eur
0,2024-01-01 00:00:00+00:00,Austria,Medical Devices,No,12,34,2118.42,1580.41,538.01
1,2024-01-01 00:00:00+00:00,Austria,Medical Devices,Yes,1,1,52.42,49.15,3.27
2,2024-01-01 00:00:00+00:00,Austria,OTC,No,64,576,6653.19,4727.64,1925.55
3,2024-01-01 00:00:00+00:00,Austria,OTC,Yes,11,114,1023.63,790.54,233.09
4,2024-01-01 00:00:00+00:00,Austria,Personal Care,No,28,227,3614.48,2375.84,1238.64


## Paso 5. Comprobar que no se ha perdido nada

Tres comprobaciones: que no hay filas repetidas (para que InfluxDB no sobrescriba), que los totales coinciden con el original, y que no hay nulos.

In [8]:
clave = ["Fecha"] + columnas_tag
print("Duplicados diario :", diario.duplicated(clave).sum())
print("Duplicados mensual:", mensual.duplicated(clave).sum())

print("Ingresos originales:", round(fact["RevenueEUR"].sum(), 2))
print("Ingresos diario    :", round(diario["ingresos_eur"].sum(), 2))
print("Ingresos mensual   :", round(mensual["ingresos_eur"].sum(), 2))

print("Nulos diario :", diario.isna().sum().sum())
print("Nulos mensual:", mensual.isna().sum().sum())

Duplicados diario : 0
Duplicados mensual: 0
Ingresos originales: 8633977.31
Ingresos diario    : 8633977.31
Ingresos mensual   : 8633977.31
Nulos diario : 0
Nulos mensual: 0


**Cómo leerlo:** los duplicados deben ser `0`, los tres totales de ingresos deben ser iguales y los nulos `0`. Si algo no cuadra, no sigas: revisa los pasos anteriores.

## Paso 6. Guardar los datos en CSV

Esto sirve si luego la subida se hace con Node-RED (Tarea 2), que leería estos archivos.

In [10]:
diario.to_csv("ventas_diarias.csv", index=False)
mensual.to_csv("ventas_mensuales.csv", index=False)

## Paso 7 (opcional). Subir directamente a InfluxDB desde Jupyter

Hazlo solo si tu profesor acepta la subida desde Jupyter. Antes instala la librería (en una celda): `!pip install influxdb-client`

Necesitas tres datos de tu InfluxDB: la dirección, la **organización** y el **token** (Load Data → API Tokens). **No compartas ni subas el token a ningún sitio.**

In [16]:
from influxdb_client import InfluxDBClient
from influxdb_client.client.write_api import SYNCHRONOUS

with open("/home/aimar/influx_token.txt") as f:
    my_token = f.read().strip()

client = InfluxDBClient(url="http://localhost:8086", token=my_token, org="somorrostro")
write_client = client.write_api(write_options=SYNCHRONOUS)

In [23]:
def subir(df, bucket, measurement="ventas", trozo=5000):
    df = df.copy()
    df[columnas_field] = df[columnas_field].astype(float)
    df[columnas_tag] = df[columnas_tag].astype(str)
    df["Fecha"] = pd.to_datetime(df["Fecha"], utc=True)
    df = df.set_index("Fecha")[columnas_tag + columnas_field]
    for i in range(0, len(df), trozo):
        write_client.write(bucket=bucket, org="somorrostro", record=df.iloc[i:i+trozo],
                           data_frame_measurement_name=measurement,
                           data_frame_tag_columns=columnas_tag)
    print(f"Subidas {len(df)} filas a {bucket}")

In [24]:
subir(diario, "ventas_diarias")
subir(mensual, "ventas_mensuales")

Subidas 29435 filas a ventas_diarias
Subidas 1908 filas a ventas_mensuales


## Comprobar en InfluxDB

En la web de InfluxDB (`http://localhost:8086`) ve a **Data Explorer**, elige el bucket y el measurement `ventas`, y marca un field (por ejemplo `margen_eur`). Si ves datos, está bien subido. Pon el rango de tiempo en *Past 5 years* o similar, porque son datos de 2024 y 2025.